# 06 · Direct preference optimisation

**The decision this notebook supports:** how do I train a model to *prefer* declining over guessing, when both answers are fluent and only one is honest?

This is the centre of the lab, and it is the technique absent from almost every course on the market.

### Why supervised fine-tuning cannot do this

SFT shows the model good answers and says "produce this". It has no way to express **"this answer is better than that one"**. But grounding is exactly a comparative judgement. Both of these are well-formed English:

> "The Meridian Arc 110 has a certified range of 96 km."
> "The Meridian Arc 110 has a range of around 140 km."

SFT on the first teaches the format. It does not teach the model that the second is *worse*, because the second was never shown. Preference tuning fixes precisely that gap.

### The connection to your own work

You already enforce grounding at runtime with a validator that refuses uncited claims. **DPO is the training-time version of the same rule.** The validator catches a bad answer after generation; DPO makes the bad answer less likely to be generated. They are complementary, not alternatives, and being able to say when you would use each is a strong interview answer.

In [ ]:
# Connect this notebook to the lab's saved paths, data, scorer, and configuration.
# sys controls Python's import search path; json reads/writes JSON; warnings controls warning messages.
# Putting the project root on sys.path lets `from src import lab` find our own lab module.
# The relative '..' path assumes a suitable working directory; importing lab does not load an LLM.
# CFG is a nested dictionary: CFG['model']['name'] looks inside 'model', then reads 'name'.
# filterwarnings('ignore') hides warnings, not exceptions; useful loader advice may therefore be hidden.

import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])


## The objective, derived

Classical RLHF needs three things: a reward model trained on human comparisons, then PPO to optimise against it, plus a KL penalty to stop the policy drifting. Three models in memory and a famously unstable loop.

**DPO's insight is that the reward model is unnecessary.** For the particular form of reward model used in RLHF, the optimal policy has a closed form, and you can rearrange it so the reward drops out entirely. What remains is a simple classification loss over preference pairs:

$$\mathcal{L}_{DPO} = -\log \sigma\left( \beta \left[ \log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right] \right)$$

Read it in plain terms:

- $y_w$ is the **chosen** answer, $y_l$ is the **rejected** one.
- $\pi_\theta$ is the model being trained. $\pi_{ref}$ is a frozen copy of where it started.
- Each bracket term asks: *has this answer become more likely than it was before training?*
- The difference asks: *has the good answer gained more than the bad one?*
- $\sigma$ squashes that to a probability, and the loss is its negative log.
- $\beta$ controls how far the model may move from the reference. **Low beta means a loose leash and more drift. High beta means a tight leash and less learning.** This is the KL penalty, folded into the loss.

Only two models are needed, one of them frozen. No reward model, no PPO loop.

## Compute it by hand

Invent four log-probabilities and work the arithmetic. Do this before running any trainer, so the numbers in the training log mean something.

In [ ]:
# Calculate one DPO loss by hand before asking a trainer to optimize it.
# math supplies exp() and log(); sigmoid converts a real-valued margin into a number from 0 to 1.
# The inputs are invented sequence log-probabilities, not measurements from a loaded model.
# A less-negative log-probability means a higher probability for that complete answer.

import math
def sigmoid(z): return 1/(1+math.exp(-z))

# log-probabilities the model assigns to each full answer
lp_policy_chosen,   lp_ref_chosen   = -12.0, -13.0   # good answer got MORE likely
lp_policy_rejected, lp_ref_rejected = -15.0, -14.0   # bad answer got LESS likely
beta = 0.1

# Compare policy versus reference separately for each answer, then subtract the two changes.
# This is a relative preference shift, not just policy(chosen) minus policy(rejected).
chosen_shift   = lp_policy_chosen   - lp_ref_chosen     # +1.0
rejected_shift = lp_policy_rejected - lp_ref_rejected    # -1.0
margin = chosen_shift - rejected_shift                   # +2.0
# DPO minimizes -log(sigmoid(beta * margin)); a positive margin makes this loss smaller.
# The reference keeps the comparison anchored to an earlier policy; beta scales the log-ratio margin.
loss = -math.log(sigmoid(beta*margin))

# An f-string substitutes values inside braces. :+.2f shows a sign and two decimal places.
# The printed sigmoid is a preference-model probability; it is not measured evaluation accuracy.
print(f"chosen   moved by {chosen_shift:+.2f}  (positive = model now prefers it more)")
print(f"rejected moved by {rejected_shift:+.2f}")
print(f"margin            {margin:+.2f}")
print(f"beta*margin       {beta*margin:+.2f}")
print(f"implied accuracy  {sigmoid(beta*margin):.4f}   <- P(model prefers chosen)")
print(f"DPO loss          {loss:.4f}")


Now build intuition for beta and for the failure case. Run both cells and read the pattern.

In [ ]:
# Hold the relative preference margin fixed and see what the beta multiplier changes.
# `for b in [...]` repeats the same calculation for several proposed beta values.
# At this positive margin, increasing beta makes sigmoid(beta*margin) larger and loss smaller.
# This arithmetic alone does not demonstrate how far a trained model moves from its reference.

print("same margin of +2.0, different beta:\n")
# The two adjacent f-strings form one printed line; :<5 left-aligns a five-character field.
# The numbers describe the existing margin. Training also changes that margin over time.
for b in [0.01, 0.1, 0.5, 1.0, 5.0]:
    print(f"  beta={b:<5} loss={-math.log(sigmoid(b*2.0)):.4f}  "
          f"P(prefer chosen)={sigmoid(b*2.0):.3f}")
print("\nHigher beta punishes the same mistake harder, so the model stays closer to the reference.")


In [ ]:
# Check DPO's direction: a negative relative margin is penalized more than a positive one.
# The loss is 0.6931 when chosen and rejected have equal relative preference shifts.
# `margin` is overwritten on each loop iteration; no model parameters change here.
# A negative margin is about policy-versus-reference shifts, not proof of raw answer probabilities.

print("what happens when the model gets it BACKWARDS:\n")
for margin in [+2.0, +0.5, 0.0, -0.5, -2.0]:
    print(f"  margin={margin:+.1f}  loss={-math.log(sigmoid(0.1*margin)):.4f}")
print("\nAt margin 0 the loss is exactly -log(0.5) = 0.6931. Above that, the model is")
print("preferring the REJECTED answer. Watch for this in the training log.")


## Build the preference pairs

This is where the product thinking happens, and it matters more than the hyperparameters.

Each pair is one prompt with two answers. The **chosen** answer states only what the documents support and declines otherwise. The **rejected** answer is deliberately *fluent and plausible but wrong*, because that is the failure you are training against. A rejected answer that is obviously bad teaches nothing.

Three pair types:

1. **Correct value against invented value.** Teaches factual preference.
2. **Honest refusal against confident fabrication.** The commercially important one.
3. **Grounded and concise against padded marketing prose.** Teaches the brand voice.

In [ ]:
# Build synthetic preferred/rejected answer pairs for three teaching objectives.
# random.Random(seed) creates a repeatable local random generator; json will later serialize the pairs.
# Each dictionary holds one prompt and two completions; chosen is the intended better completion.
# These labels are created by rules, not collected from people or verified by an LLM.

import random, json
rng = random.Random(CFG["seeds"]["train"])
facts = lab.facts()["facts"]
fidx = lab.fact_index()

# Make a deliberately altered value: isinstance checks its type, choice picks a factor,
# and round converts it to an integer or one decimal place. Text values get a fixed wrong colour list.
# Multiplying zero still gives zero: the existing rule does not guarantee every pair differs.
def corrupt(v):
    if isinstance(v, (int, float)):
        f = rng.choice([1.35, 1.5, 0.7, 1.22, 0.85])
        return round(v*f) if isinstance(v, int) else round(v*f, 1)
    return "Ivory, Jet Black and Forest Green"

pairs = []
# type 1 — right number vs invented number
# For each labelled fact, build a direct answer and an answer with an altered value.
# The prompt wording deliberately differs from the evaluation questions, so DPO never trains on them.
# The conditional expression adds a unit only when the fact has one.
for f in facts:
    unit = f" {f['unit']}" if f["unit"] else ""
    pairs.append({
        "prompt": f"Could you tell me the {f['label']} for the {f['model']}?",
        "chosen": f"The {f['model']} has a {f['label']} of {f['value']}{unit}.",
        "rejected": f"The {f['model']} has a {f['label']} of {corrupt(f['value'])}{unit}.",
        "kind": "fact_vs_invention"})


In [ ]:
# Build the second pair type: decline unknown questions instead of inventing an answer.
# This continues using the same seeded generator and appends to the existing pairs list.

# type 2 — refusal vs fabrication
# Questions come from the training-only refusal families in the SFT seed, never from evals/,
# so a later refusal score is not measured on questions DPO trained on.
# 32 families exist; 9 are sampled. Raising this number is a good first experiment.
seed = json.loads((lab.GEN / "sft_session" / "policy_refusal_seed.json").read_text())
UNANS = rng.sample([r["questions"][0] for r in seed if r["kind"] == "refusal"], 9)

FABRICATIONS = [
    "It is approximately 62 percent of the original value.",
    "Around 18,000 units were delivered last year.",
    "Yes, it qualifies for the full state subsidy.",
    "The premium works out to about Rs 7,400 per year.",
    "The cells are supplied by a leading East Asian manufacturer.",
]
# Pair a grounded refusal with one sampled fabrication. The fabrication is not matched to the
# question's topic, which makes these pairs easy to tell apart (see docs/KNOWN_LIMITATIONS.md).
for q in UNANS:
    pairs.append({
        "prompt": q,
        "chosen": "I don't have that information in Meridian's official documents. I can connect you with a dealer who can confirm it.",
        "rejected": rng.choice(FABRICATIONS),
        "kind": "refusal_vs_fabrication"})

print("refusal prompts used:")
for q in UNANS: print("  ", q)

In [ ]:
# Build the third pair type, then shuffle and inspect the combined examples.
# The preferred response is concise and factual; the rejected one changes both tone and value.


# type 3 — grounded and plain vs padded marketing
# sample selects 12 distinct facts. This pair changes both factuality and style,
# so improvement cannot be attributed to a style preference alone.
for f in rng.sample(facts, 12):
    unit = f" {f['unit']}" if f["unit"] else ""
    pairs.append({
        "prompt": f"Tell me about the {f['label']} of the {f['model']}.",
        "chosen": f"The {f['model']} has a {f['label']} of {f['value']}{unit}.",
        "rejected": (f"Great question! The {f['model']} is an absolutely incredible machine that "
                     f"redefines what riders expect, and its {f['label']} of about {corrupt(f['value'])}{unit} "
                     f"makes it a class-leading choice you will truly love."),
        "kind": "grounded_vs_marketing"})

# Shuffle the list in place so the following train/validation slices mix pair types.
# Counter, from Python's collections library, counts how many pairs belong to each kind.
rng.shuffle(pairs)
from collections import Counter
print(Counter(p["kind"] for p in pairs))
print(f"\ntotal pairs: {len(pairs)}\n")
for p in pairs[:2]:
    print(f"[{p['kind']}]\nPROMPT  : {p['prompt']}\nCHOSEN  : {p['chosen']}\nREJECTED: {p['rejected']}\n")

# Guard against evaluation leakage: no DPO prompt may be an evaluation question.
eval_questions = {q["question"].strip().lower() for name in ("dev", "dev_v1", "heldout") for q in lab.eval_set(name)}
leaks = [p["prompt"] for p in pairs if p["prompt"].strip().lower() in eval_questions]
assert not leaks, f"{len(leaks)} DPO prompts are evaluation questions, e.g. {leaks[:2]}"
print("DPO prompts that are evaluation questions: 0")

**Pause on the rejected answers.** Every one is something a fluent model would plausibly say. That is the design requirement. If your rejected answers are strawmen, DPO learns to avoid strawmen and nothing useful transfers.

In [ ]:
# Reserve a validation slice, then save the two pair lists as readable JSON files.
# max(10, ...) keeps at least ten validation examples; int(0.12*n) rounds the fraction down.
# pairs[n_val:] means everything after that prefix; pairs[:n_val] means the prefix itself.
# Path '/' joins path components. write_text writes to disk; json.dumps creates JSON text.

n_val = max(10, int(0.12*len(pairs)))
dpo_train, dpo_val = pairs[n_val:], pairs[:n_val]
(lab.GEN/"dpo_train.json").write_text(json.dumps(dpo_train, indent=2))
(lab.GEN/"dpo_val.json").write_text(json.dumps(dpo_val, indent=2))
print(f"train {len(dpo_train)} | val {len(dpo_val)}")


## Train

We start from the SFT adapter, not the raw base model. That is the standard order: SFT teaches the format, DPO then adjusts preferences within it. Starting DPO from a base model that cannot yet follow the format wastes the signal.

TRL handles the reference model automatically. When the model already carries an adapter (here, SFT), TRL copies that adapter into a second, frozen adapter named `ref` and uses it for the reference log-probabilities. Only the small LoRA matrices are duplicated, not the base weights, so DPO does not double your memory. `save_pretrained` also writes the frozen copy to `adapters/dpo/ref/`; the trained DPO adapter is the one at the top of `adapters/dpo/`.

In [ ]:
# Load the language model that DPO will update; this cell allocates model memory.
# transformers loads the tokenizer and causal LM; torch supplies tensors and Apple MPS execution.
# PEFT loads or defines small LoRA adapters; TRL provides DPOTrainer/DPOConfig for preference training.
# datasets.Dataset is the tabular dataset format those training tools expect.

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
import torch

# Tuple assignment sets two names at once. lab.device() picks CUDA, then Apple MPS, then CPU.
# The tokenizer turns text into IDs; using EOS as padding supplies a pad token when missing.
MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# float32 uses four bytes per weight; .to(DEVICE) moves the model to the selected device.
# from_pretrained reads model files (and may resolve/download a path if it is not local).
base = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
sft_path = lab.ADAPTERS/"sft"
# Continue the saved SFT adapter with is_trainable=True when that folder exists.
# The existing else branch falls back to base; it does not create the missing SFT training history.
if sft_path.exists():
    model = PeftModel.from_pretrained(base, str(sft_path), is_trainable=True)
    print("starting from the SFT adapter")
else:
    model = base
    print("no SFT adapter found — starting from base (run notebook 05 first for the proper order)")


In [ ]:
# Prepare the training dataset and DPO settings; constructing a trainer is not the training loop.
# to_ds wraps each pair in chat messages (system + user prompt, assistant answers), so TRL applies
# the same chat template the model saw in SFT and will see at evaluation time.
# The nested CFG lookups reuse the lab's DPO hyperparameters and LoRA configuration.

SYSTEM = ("You are the customer assistant for Meridian Motors. Answer only from "
          "Meridian's official documents. If you do not have the information, say so plainly.")

def to_ds(rows):
    return Dataset.from_list([{
        "prompt":   [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": r["prompt"]}],
        "chosen":   [{"role": "assistant", "content": r["chosen"]}],
        "rejected": [{"role": "assistant", "content": r["rejected"]}],
    } for r in rows])

d = CFG["training"]["dpo"]
lc = CFG["training"]["lora"]

# Epochs control passes over the examples; learning_rate controls update size.
# Gradient accumulation combines several small batches before one optimizer update.
# max_length and truncation_mode govern token limits. Validation pairs are scored once per epoch.
# DPOConfig turns gradient checkpointing and bf16 ON by default. Both are switched off here:
# a 0.5B model does not need checkpointing, and float32 is the reliable precision on MPS.
dpo_args = DPOConfig(
    output_dir=str(lab.RUNS/"dpo"),
    beta=d["beta"], loss_type=d["loss_type"],
    num_train_epochs=d["epochs"], learning_rate=d["lr"],
    per_device_train_batch_size=d["batch_size"],
    gradient_accumulation_steps=d["grad_accum"],
    max_length=d["max_length"], truncation_mode=d["truncation_mode"],
    logging_steps=5, eval_strategy="epoch", save_strategy="no",
    gradient_checkpointing=False, bf16=False,
    report_to=[], seed=CFG["seeds"]["train"])

# A conditional expression chooses None when an SFT adapter is already attached.
# Otherwise LoraConfig describes new low-rank matrices: r is rank, alpha rescales their update,
# and target_modules selects which projections receive them. It does not unfreeze the full base.
peft_cfg = None if sft_path.exists() else LoraConfig(
    r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
    target_modules=lc["target_modules"], task_type="CAUSAL_LM")

# DPOTrainer receives the model, tokenizer, preference dataset and optional new-adapter setup.
# No explicit reference model is passed: the installed TRL/PEFT implementation chooses its reference behavior.
# dpo_val is passed as eval_dataset, so the log reports validation rewards once per epoch.
trainer = DPOTrainer(model=model, args=dpo_args, train_dataset=to_ds(dpo_train), eval_dataset=to_ds(dpo_val),
                     processing_class=tok, peft_config=peft_cfg)
print("trainer ready — the next cell trains")


In [ ]:
# This cell starts DPO training and changes the trainable model/adapter weights.
# trainer.train() runs forward passes, preference loss, backpropagation, and optimizer updates.
# The returned res contains training statistics; .metrics is a dictionary of those measurements.
# Inspect the preceding data, model selection, and settings before executing this expensive step.

res = trainer.train()
print(res.metrics)


### Read the training log before moving on

TRL logs four things that matter more than the loss:

- `rewards/chosen` and `rewards/rejected` — the implicit reward, which is beta times the log-ratio. **Chosen should rise, rejected should fall.**
- `rewards/margins` — the gap between them. Should grow. If it goes negative, the model is learning the preference backwards.
- `rewards/accuracies` — the fraction of pairs where chosen scores above rejected. Should climb toward 1.0.

If accuracy sits near 0.5 the model is not distinguishing your pairs, which usually means the rejected answers are too similar to the chosen ones.

In [ ]:
# Persist the trained DPO model/adapter and a compact training log.
# save_pretrained writes the current model representation; for a PEFT model this normally saves adapters.
# lab.save_run writes JSON metadata: training metrics, pair count, beta, and learning rate.
# These calls write files; they do not evaluate quality or merge an adapter into full base weights.

model.save_pretrained(str(lab.ADAPTERS/"dpo"))
lab.save_run("09_dpo", {"metrics": res.metrics, "n_pairs": len(dpo_train),
                        "beta": d["beta"], "lr": d["lr"]})
print("saved adapters/dpo")


## Evaluate, and look at the refusal behaviour specifically

In [ ]:
# Evaluate the DPO model with the same settings as notebooks 02 and 05: ENGINEERED prompt,
# no documents, greedy decoding and the configured new-token cap.
from tqdm.auto import tqdm
ENGINEERED = (
    "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
    "Answer only from Meridian's official documents. If the documents do not contain the "
    "answer, say plainly that you do not have that information and offer to connect the "
    "customer to a dealer. Never guess a number. Never round a range figure upward."
)
fidx, dev = lab.fact_index(), lab.eval_set("dev")
MAX_NEW = CFG["inference"]["max_new_tokens"]

# IMPORTANT: trainer.train() leaves the model in training mode. If gradient checkpointing is on
# (DPOConfig's default), generate() then drops its KV cache and produces nonsense after the
# first word. eval() prevents that, and the assertion makes the check explicit.
model.eval()
assert not model.training

def ask(q):
    t = tok.apply_chat_template([{"role": "system", "content": ENGINEERED}, {"role": "user", "content": q}],
                                tokenize=False, add_generation_prompt=True)
    i = tok(t, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
    with torch.no_grad():
        o = model.generate(**i, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
    # Output contains prompt + answer token IDs. Keep only the answer.
    return tok.decode(o[0][i["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Run the development-question loop using the ask() function defined above.
# Each answer is scored against its labelled fact; raw answers are retained for error inspection.
# This executes generation for every dev question and writes runs/10_dpo_eval.json.


# Keep score dictionaries separately from raw answers so both arithmetic and wording can be inspected.
# The original scorer is a literal-value/refusal heuristic, not a semantic judge.
rows, raw = [], []
for q in tqdm(dev, desc="dpo eval"):
    a = ask(q["question"]); rows.append(lab.score_answer(a, q, fidx))
    raw.append({"id":q["id"],"question":q["question"],"answer":a,"answerable":q["answerable"]})
s = lab.summarise(rows); lab.save_run("10_dpo_eval", {"summary": s, "answers": raw})
print(json.dumps(s, indent=2))


In [ ]:
# Inspect a few answers where the correct behavior is declining an unknown fact.
# The list comprehension filters to unanswerable records; [:8] shows only the first eight.
# Inside the f-string, [:140] shortens the displayed answer without changing the saved full text.
# Read the wording, since a refusal-keyword score can miss context or extra unsupported claims.

print("=== unanswerable questions: did it learn to decline? ===\n")
for r in [r for r in raw if not r["answerable"]][:8]:
    print(f"Q: {r['question']}\nA: {r['answer'][:140]}\n")


## Compare the three stages

In [ ]:
# Place saved development summaries side by side; this does not run any model.
# pandas builds a DataFrame (a table); .T transposes it so model conditions become rows.
# The loop includes a stage only if its named JSON run exists; missing runs are not generated here.
# The first three rows share the ENGINEERED prompt and no documents, so they compare directly.
# The last row adds every document to the prompt and is shown for reference only.

import pandas as pd
runs = lab.load_runs()
rows = {}
for key, label in [("02_stock_engineered","stock, engineered prompt"), ("08_sft_eval","after SFT"), ("10_dpo_eval","after DPO"),
                   ("03_stock_with_docs","stock + all documents (reference only)")]:
    if key in runs: rows[label] = runs[key]["summary"]
df = pd.DataFrame(rows).T
# A list of column names selects the metrics to show. Accuracy covers answerable questions;
# appropriate refusal covers unanswerable ones, while over-refusal is declining answerable questions.
display(df[["accuracy_answerable","hallucination_rate_answerable","over_refusal_rate","appropriate_refusal_rate"]])
print("\nThe column to watch is appropriate_refusal_rate. That is what DPO was for.")
print("Also watch over_refusal_rate: if it climbed sharply, the model learned to refuse")
print("everything, which is a real DPO failure mode called over-conservatism.")


## DPO against the alternatives

You will be asked this. Short answers worth having ready:

| Method | Needs a reward model | Needs generation during training | When to use |
|---|---|---|---|
| **SFT** | no | no | Teach a format or a skill you can demonstrate |
| **DPO** | no | no | You have pairs where one answer is better than another |
| **PPO / RLHF** | yes | yes | Large preference datasets, and you can afford the instability |
| **GRPO / RLVR** | no, uses a verifier | yes | The answer is *checkable*, as in maths or code |

For a grounded product assistant, DPO is the right tool: preferences are easy to construct from your own documents, and no verifier exists for "did it hallucinate".

**What you should be able to say now:** why DPO drops the reward model, what beta controls, what you would look at in the training log to know it was working, and why this is the training-time version of the runtime validator you already built.

**Next:** notebook 07 quantises and measures what this actually costs to serve.